# Module 6: LLM Optimization

**Goal:** Cut cost and latency without sacrificing quality — the core engineering challenge after you have a working system.

**What you'll learn:**
1. Token efficiency: prompt compression without quality loss
2. Multi-level caching (exact-match → semantic → prefix)
3. Intelligent model routing (cheap vs. expensive)
4. Response length control
5. Measuring optimization impact rigorously

> **Key mindset:** Always measure before optimizing. A 10-line change that cuts cost 60% is useless if quality drops 30%.

---

## 0. Setup

In [ ]:
%pip install -q openai tiktoken numpy python-dotenv

In [ ]:
import os, time, json
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

import tiktoken
import numpy as np
from openai import OpenAI

client = OpenAI()
enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def llm_call(prompt: str, model="gpt-4o-mini", max_tokens=None) -> dict:
    """Returns response + usage stats."""
    kwargs = dict(model=model, messages=[{"role": "user", "content": prompt}], temperature=0)
    if max_tokens:
        kwargs["max_tokens"] = max_tokens
    start = time.time()
    resp = client.chat.completions.create(**kwargs)
    return {
        "content": resp.choices[0].message.content,
        "input_tokens": resp.usage.prompt_tokens,
        "output_tokens": resp.usage.completion_tokens,
        "latency_ms": round((time.time() - start) * 1000),
    }

# Pricing ($/token)
PRICE = {"gpt-4o-mini": (0.15e-6, 0.60e-6), "gpt-4o": (2.5e-6, 10e-6)}

def cost(result: dict, model="gpt-4o-mini") -> float:
    ip, op = PRICE[model]
    return result["input_tokens"] * ip + result["output_tokens"] * op

print("✅ Ready")

---
## 1. Baseline: Measure Before Optimizing

In [ ]:
# A verbose, unoptimised prompt
verbose_prompt = """
Hello! I hope you're doing well today. I am writing to you because I have an important question 
that I need your help with. I would really appreciate it if you could take the time to carefully 
consider my question and provide me with a thorough and comprehensive answer. The question I have 
is about the Python programming language, specifically regarding how Python handles list 
comprehensions and what the syntax looks like. I would like to understand this concept better. 
Please provide a detailed explanation with examples if possible.
"""

result = llm_call(verbose_prompt)
print(f"Verbose prompt:")
print(f"  Input tokens:  {result['input_tokens']}")
print(f"  Output tokens: {result['output_tokens']}")
print(f"  Latency:       {result['latency_ms']}ms")
print(f"  Cost:          ${cost(result):.6f}")

---
## 2. Prompt Compression

In [ ]:
# Compressed version — same meaning, fewer tokens
compressed_prompt = "Explain Python list comprehensions with 2 examples."

verbose_tokens = count_tokens(verbose_prompt)
compressed_tokens = count_tokens(compressed_prompt)
reduction = (1 - compressed_tokens / verbose_tokens) * 100

print(f"Token comparison:")
print(f"  Verbose:    {verbose_tokens} tokens")
print(f"  Compressed: {compressed_tokens} tokens")
print(f"  Reduction:  {reduction:.0f}%")

result_compressed = llm_call(compressed_prompt)
print(f"\nCompressed result:")
print(f"  Input tokens:  {result_compressed['input_tokens']}")
print(f"  Cost:          ${cost(result_compressed):.6f}")
print(f"  Cost savings:  {(1 - cost(result_compressed)/cost(result))*100:.0f}%")

In [ ]:
def compress_prompt(verbose: str, target_reduction: float = 0.5) -> str:
    """Use an LLM to compress a prompt while preserving intent."""
    meta_prompt = f"""Compress this prompt to ~{int((1-target_reduction)*100)}% of its length.
Preserve all instructions and constraints. Remove filler, pleasantries, redundancy.
Output only the compressed prompt, nothing else.

Prompt to compress:
{verbose}"""
    result = llm_call(meta_prompt)
    return result["content"]

auto_compressed = compress_prompt(verbose_prompt)
print(f"Auto-compressed ({count_tokens(auto_compressed)} tokens):")
print(auto_compressed)

---
## 3. Intelligent Model Routing

Route simple queries to cheap models, complex ones to expensive models.

In [ ]:
def classify_query_complexity(query: str) -> str:
    """Route to 'simple' or 'complex'."""
    simple_indicators = [
        len(query.split()) < 15,           # Short query
        query.endswith("?") and "?" not in query[:-1],  # Single question
        any(q in query.lower() for q in ["what is", "define", "who is", "when was"]),
        not any(k in query.lower() for k in ["explain", "compare", "analyze", "design", "write code"]),
    ]
    return "simple" if sum(simple_indicators) >= 2 else "complex"


def smart_route(query: str) -> dict:
    """Route query to appropriate model and return result + routing decision."""
    complexity = classify_query_complexity(query)
    model = "gpt-4o-mini" if complexity == "simple" else "gpt-4o"
    result = llm_call(query, model=model)
    result["model_used"] = model
    result["complexity"] = complexity
    result["cost"] = cost(result, model)
    return result


test_queries = [
    "What is Python?",
    "Who invented the internet?",
    "Design a distributed rate limiter for 10M requests/day. Consider CAP theorem trade-offs and compare Redis vs. DynamoDB approaches.",
    "Compare transformer vs. RNN architectures for sequence modelling tasks with long-range dependencies.",
]

total_smart_cost = 0
total_naive_cost = 0  # Always gpt-4o

print(f"{'Query':50s} {'Complexity':10s} {'Model':15s} {'Cost':10s}")
print("-" * 90)
for q in test_queries:
    result = smart_route(q)
    naive = llm_call(q, model="gpt-4o")
    total_smart_cost += result["cost"]
    total_naive_cost += cost(naive, "gpt-4o")
    print(f"{q[:50]:50s} {result['complexity']:10s} {result['model_used']:15s} ${result['cost']:.6f}")

savings = (1 - total_smart_cost / total_naive_cost) * 100
print(f"\nSmart routing cost:  ${total_smart_cost:.5f}")
print(f"Always GPT-4o cost:  ${total_naive_cost:.5f}")
print(f"Savings:             {savings:.0f}%")

---
## 4. Response Length Control

In [ ]:
question = "What is Docker and why should I use it?"

length_variants = [
    ("No constraint",   question),
    ("1 sentence",      f"{question} Answer in exactly 1 sentence."),
    ("3 bullets",       f"{question} Answer in exactly 3 bullet points."),
    ("max_tokens=50",   question),  # We'll pass max_tokens separately
]

print(f"{'Variant':20s} {'Tokens out':12s} {'Cost':12s} Preview")
print("-" * 80)
for name, prompt in length_variants:
    kwargs = {"max_tokens": 50} if name == "max_tokens=50" else {}
    r = llm_call(prompt, **kwargs)
    print(f"{name:20s} {r['output_tokens']:12d} ${cost(r):.6f}   {r['content'][:50]}...")

---
## 5. Optimization Impact Summary

In [ ]:
print("""
Optimization Summary
====================

Technique               Typical Cost Reduction    Complexity
──────────────────────  ────────────────────────  ──────────
Prompt compression      20-50%                    Low
Exact-match cache       5-15% (for repeat queries) Low
Semantic cache          10-30%                    Medium
Model routing           30-70%                    Medium
Response length control 20-40%                    Low
Batching requests       15-25% (latency, not cost) Medium
Quantized local model   60-90% (infra cost swap)  High

ORDER OF OPERATIONS:
1. Measure baseline cost & quality
2. Compress prompts (free wins)
3. Add semantic cache (high ROI if repeat queries exist)
4. Add model routing (biggest savings for mixed workloads)
5. Only then consider self-hosting
""")

---
## 🧪 Exercises

1. **Compression quality check**: Compress 10 prompts automatically, then evaluate whether the answers are equally good using the LLM-as-judge from Module 04. What's the quality-cost trade-off?
2. **Improve the router**: The current router uses heuristics. Replace `classify_query_complexity` with an LLM classifier (prompt: "Is this query simple or complex? Output one word."). Is it more accurate? At what added cost?
3. **Cache hit rate experiment**: Build a dataset of 50 questions, where 30 are near-duplicates of 10 unique questions. What threshold gives the best precision-recall for cache hits?
4. **Real-world simulation**: Grab 100 questions from a public Q&A dataset. Apply all 4 optimisations in sequence and measure cumulative cost reduction.

---
**Next:** [Module 07 — Agentic Workflows](../07-agentic-workflows/README.md)